# سبق 18: کرپٹوگرافک رسیدوں کے ساتھ AI ایجنٹس کی حفاظت

## عملی نوٹ بک

یہ نوٹ بک چار مشقوں کے ذریعے رہنمائی کرتی ہے:

1. کسی ایجنٹ ٹول کی کال کے لیے **اپنی پہلی رسید پر دستخط کریں** اور اس کی تصدیق کریں۔
2. **رسید میں چھیڑ چھاڑ کریں** اور تصدیق کے ناکام ہونے کو دیکھیں۔
3. **تین رسیدوں کی زنجیر بنائیں** اور زنجیر کی سالمیت کی تصدیق کریں۔
4. **مائیکروسافٹ ایجنٹ فریم ورک کے ٹول کال کو لپیٹیں** تاکہ ہر عمل کے ساتھ ایک رسید بھی جاری ہو۔

تمام کرپٹوگرافک اصل عناصر معروف لائبریریوں سے امپورٹ کیے گئے ہیں (`pynacl` ایڈ25519 کے لیے، `jcs` RFC 8785 کینونیکل JSON کے لیے، `hashlib` Python اسٹینڈرڈ لائبریری سے SHA-256 کے لیے)۔ رسید کا منطق بذات خود سادہ Python ہے جسے آپ پڑھ اور ترمیم کر سکتے ہیں۔

خلیات کو ترتیب سے چلائیں۔ ہر سیکشن مختصر اور خود مختار ہے۔


## سیٹ اپ

دو منحصرات انسٹال کریں۔ دونوں کے پاس اجازت دینے والی لائسنس (Apache-2.0 / MIT) ہیں۔


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## معاون یوٹیلٹیز

یہ دو معاون (ہیلیپر) بیس64یو آر انکوڈنگ (بغیر پیڈنگ کے) اور بے ترتیبی اشیاء کی SHA-256 ہیشنگ کو ہینڈل کرتے ہیں۔ یہ نوٹ بک کے باقی حصے کو خود رسید کے منطق پر مرکوز رکھتے ہیں۔


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## سیکشن 1: اپنی پہلی رسید پر دستخط کریں

تصور کریں کہ ہمارے ایجنٹ برائے **Contoso Travel** نے ابھی ایک صارف کے لیے سڈنی سے لاس اینجلس تک پروازیں تلاش کی ہیں۔ ہم چاہتے ہیں کہ گیٹ وے کے اس ٹول کال کے بیان کو ریکارڈ کریں تاکہ مستقبل کے آڈیٹر اسے ایک علٰیحدہ پن کی گئی جاری کنندہ کلید کے تحت تصدیق کر سکیں۔

### مرحلہ 1.1: دستخطی کلید تیار کریں

پیداوار میں، ایجنٹ کی دستخطی کلید ہارڈویئر سیکیورٹی ماڈیول (HSM)، Azure Key Vault، یا اسی طرح کے محفوظ اسٹور میں رکھی جاتی ہے۔ اس سبق کے لیے ہم میموری میں ایک تازہ کلید تیار کرتے ہیں۔


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### مرحلہ 1.2: رسید کے پے لوڈ کی تیاری

پے لوڈ میں وہ سب کچھ شامل ہوتا ہے جس کی ہم رسید سے تصدیق چاہتے ہیں: کس نے عمل کیا، کون سا آلہ استعمال کیا، کن دلائل کے ساتھ، کیا واپس آیا، کس پالیسی کے تحت، اور کب۔ ہم دلائل اور نتائج کا ہیش بناتے ہیں بجائے اس کے کہ انہیں براہ راست شامل کریں تاکہ رسید حساس مواد کو افشا نہ کرے۔


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### مرحلہ 1.3: رسید پر دستخط کریں اور اسے اسمبل کریں

تین مراحل:

1۔ جے سی ایس استعمال کرتے ہوئے پے لوڈ کو کینونکلائز کریں تاکہ دو امپلیمنٹیشنز جو ایک ہی منطقی رسید پیدا کرتی ہیں، وہ بائٹ کی سطح پر بالکل ایک جیسی بائٹس پیدا کریں۔
2۔ کینونکل بائٹس پر براہ راست Ed25519 پرائیویٹ کی سے دستخط کریں۔ PureEdDSA پیغام کو اندرونی طور پر ہیش کرتا ہے، لہٰذا ایک اضافی پِری ہیش پروٹوکول کو تبدیل کر دے گا۔

دستخط پھر اصل پے لوڈ سے منسلک کیا جاتا ہے تاکہ آخری رسید بن جائے۔


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### مرحلہ 1.4: رسید کی تصدیق کریں

تصدیق عمل کو الٹ دیتا ہے۔ ہم دستخط کو ہٹا دیتے ہیں، کینونیکل بائٹس کو دوبارہ حساب کرتے ہیں، اور دستخط کو جاری کنندہ کی پبلک کی کے خلاف چیک کرتے ہیں۔

تصدیق آف لائن ہوتی ہے: کوئی سروس کال کرنے کی ضرورت نہیں اور کوئی نیٹ ورک رسائی نہیں ہے۔ پھر بھی اسے رسید کے علاوہ ایک چیز کی ضرورت ہوتی ہے: جاری کنندہ کی پبلک کی، جو کہ آڈیٹر کے پہلے سے معتبر چینل کے ذریعے حاصل کی جاتی ہے، جیسے کہ جاری کنندہ کے شائع کردہ JWK سیٹ یا پہلے سے طے شدہ کوئی ویلیو۔ رسید کے اندر`public_key` فیلڈ وہ کردار ادا نہیں کر سکتا، کیونکہ جو بھی رسید لکھتا ہے وہ وہ فیلڈ بھی لکھتا ہے۔ سیکشن 2 اس جعلسازی کو دکھاتا ہے جسے یہ روکتا ہے۔


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


آپ کو `Receipt is valid: True`، `Pre-hashed receipt valid: False` اور `Malformed receipts rejected: True` دیکھنے چاہئیں۔ مثبت کیس سے ثابت ہوتا ہے کہ ڈائریکٹ-JCS دستخطی راستہ کام کرتا ہے؛ منفی کنٹرول دستخط-دائرہ کار قاعدہ کو عملی بناتا ہے بجائے اس کے کہ اسے نثر کے طور پر چھوڑ دیا جائے؛ آخری لائن خراب ان پٹ کو ظاہر کرتی ہے جو استثناء اٹھانے کے بجائے `False` لوٹاتی ہے۔


## سیکشن 2: رسید میں مداخلت کریں

رسیدوں کا اصل مقصد یہ ہے کہ وہ مداخلت کے ثبوت ہوں۔ آئیے اسے ثابت کریں۔

ہم بالکل ایک حرف کو رسید میں ترمیم کریں گے اور تصدیق کے ناکام ہونے کو دیکھیں گے۔


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### کیا ہوا ابھی؟

جب ہم نے `policy_id` کو تبدیل کیا، تو canonical بائٹس تبدیل ہوگئیں۔ دستخط (جو اصل canonical بائٹس پر مبنی تھا) اب میل نہیں کھاتا۔ تصدیق درست طور پر `False` لوٹاتی ہے۔

کوئی طریقہ نہیں ہے کہ دستخط شدہ فیلڈ کو، یعنی `signature` آبجیکٹ کے باہر کچھ بھی، تبدیل کیا جائے اور پھر بھی رسید جاری کنندہ کی پبلک کی کے خلاف ویریفائی ہو، جب تک کہ حملہ آور کے پاس جاری کنندہ کی پرائیویٹ کی نہ ہو۔ `signature` آبجیکٹ خود دستخط شدہ نہیں ہے: `sig` دستخط ہے، اور `verify_receipt` `public_key` کو نظر انداز کرتا ہے، اس لیے اس فیلڈ کو اکیلے ایڈیٹ کرنے سے کچھ نہیں بدلتا۔ جب تک کہ پرائیویٹ کی کسی کی والٹ میں محفوظ ہو اور ویریفائیرز جاری کنندہ کی آزادانہ طور پر شائع کردہ کی استعمال کریں، دستخط شدہ فیلڈ میں تبدیلی چھپانا ناممکن ہے۔ یہ شرط اہم ہے: اگلا سیل ایک ترمیم شدہ رسید دکھاتا ہے جو مختلف کی کے خلاف بالکل ویریفائی ہوجاتا ہے۔

خود آزمائیں: اوپر کے سیل میں `tool_name` یا `agent_id` یا `timestamp` کو تبدیل کریں اور دوبارہ چلائیں۔ دستخط شدہ فیلڈ میں ہر تبدیلی ایک غلط رسید پیدا کرتی ہے۔


### وہ چالاکی جو دستخط نہیں پکڑ پاتی

اوپر والا حملہ آور دوبارہ دستخط نہیں کرتا تھا۔ ایک زیادہ محتاط حملہ آور یہ کرے گا: رسید میں ترمیم کرے گا،
اسے خود تیار کردہ کلید کے ساتھ دستخط کرے گا، اور وہ کلید
`public_key` فیلڈ میں ڈالے گا۔ نتیجہ ایک ایسی رسید ہوگی جس کے دستخط اس کے مواد کے لیے مکمل طور پر اصلی ہوں گے۔


اسے چلائیں اور دیکھیں کہ دو چیکوں میں سے کون سا اسے پکڑتا ہے۔


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


آپ کو `True` اور پھر `False` دیکھنا چاہیے۔ تینوں `assert` لائنیں ایک 
چیک کرتی ہیں: اگر جعلسازی کبھی بھی جاری کنندہ کی کلید کے خلاف درست ثابت ہوتی ہے تو یہ سیل ایک خرابی پیدا کرتا ہے۔
جاری کنندہ کی کلید کے خلاف درست ثابت ہوتی ہے۔

جعلسازی اندرونی طور پر مطابقت رکھتی ہے۔ اس پر دستخط واقعی اس کے مواد اور اس کلید سے میل کھاتے ہیں جو یہ لے کر چلتی ہے، لہٰذا ایک توثیق کنندہ جو رسید سے کلید پڑھتا ہے اسے درست بتاتا ہے۔ رسید میں کچھ بھی غلط نظر نہیں آتا، کیونکہ
کچھ بھی اس میں غیر متناسب نہیں ہے۔



اس کو پکڑنے والی واحد چیز ایک ایسے کلید کے خلاف چیک کرنا ہے جسے حملہ آور کے پاس اثر انداز ہونے کا کوئی طریقہ نہیں تھا۔
اسی لیے `verify_receipt` کلید کو ایک دلیل کے طور پر لیتا ہے: ایک
فنکشن جس کا دستخط `verify_receipt(receipt)` ہے حفاظت سے نہیں لکھا جا سکتا،
کیونکہ واحد کلید تک جو وہ پہنچ سکتا ہے وہی ہے جو حملہ آور نے فراہم کی تھی۔

وہی رسید `sample_receipts/04_resigned_receipt.json` کے طور پر محفوظ کی جاتی ہے اگر آپ
اسے نوٹ بک کے باہر معائنہ کرنا چاہتے ہیں۔


## سیکشن 3: ریسیپٹس کو آپس میں جوڑنا  

ایک واحد رسید ایک عمل کی حفاظت کرتی ہے۔ زیادہ تر ایجنٹ متعدد اعمال انجام دیتے ہیں۔ پورے سلسلے کو چھیڑ چھاڑ سے محفوظ بنانے کے لیے، ہم ہر رسید کو پچھلی رسید کے ساتھ جوڑتے ہیں، پچھلی رسید کا ہیش نئی رسید کے پیلوڈ میں شامل کرکے۔  

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```
  
اگر کوئی پیش کیے گئے سلسلے میں کسی رسید کو ہٹا دے یا ترتیب بدلے، تو اگلا لنک ٹوٹ جائے گا۔ کٹاؤ کا پتہ لگانے یا دستخط کنندہ کے کنٹرول شدہ لاحقے کی دوبارہ تحریر کے لیے بھی ایک مقررہ موجودہ سر، بیرونی چیک پوائنٹ، یا مستند آبادی کے عزم کی ضرورت ہوتی ہے۔  


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


اب زنجیر کو اس طرح توڑیں کہ درمیان والی رسید میں چھیڑ چھاڑ کریں اور دوبارہ تصدیق کریں۔ چھیڑی گئی رسید اپنی دستخطی جانچ میں ناکام ہوتی ہے، اور اگلی رسید اپنی چین-لنک جانچ میں ناکام ہوتی ہے (کیونکہ اس کا `previous_receipt_hash` اب تبدیل شدہ وسطی رسید کے ہیش سے مطابقت نہیں رکھتا)۔


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


رسید 0 ابھی بھی تصدیق کرتا ہے (یہ تبدیل نہیں ہوئی تھی اور اس کا کوئی پیشرو نہیں جس پر انحصار کیا جا سکے)۔ رسید 1 اپنی دستخطی جانچ میں ناکام ہو جاتی ہے کیونکہ ہم نے `tool_args_hash` کو تبدیل کیا۔ رسید 2 اپنی زنجیری لنک جانچ میں ناکام ہو جاتی ہے کیونکہ اس کا `previous_receipt_hash` اصل (اب تبدیل شدہ) رسید 1 کے مقابلے میں حساب کیا گیا تھا۔

یہاں تک کہ اگر ایک حملہ آور تبدیل شدہ رسید 1 پر دوبارہ دستخط کرے، نیا دستخط اجرا کنندہ کی کلید کے خلاف ناکام ہو جائے گا جب تک کہ ان کے پاس اجرا کنندہ کی نجی کلید نہ ہو (اپنی کلید سے دوبارہ دستخط ناکام ہوتا ہے، جیسا کہ سیکشن 2 نے دکھایا)، اور رسید 2 میں زنجیری لنک کا تعارض اب بھی چالاکی کو ظاہر کرے گا۔ تبدیلی کو چھپانے کے لیے، حملہ آور کو تبدیلی کے مقام سے لے کر ہر رسید پر دوبارہ دستخط کرنا پڑے گا، جو اجرا کنندہ کی نجی کلید کے قبضے کا تقاضا کرتا ہے۔


## سیکشن 4: ایجنٹ ٹول کال کو رسید دستخط کے ساتھ لپیٹنا

حقیقی تعیناتی میں، آپ نہیں چاہتے کہ ہر ایجنٹ مصنف کو `make_receipt` کال کرنا یاد رہے۔ آپ چاہتے ہیں کہ ہر ٹول کال کے لیے رسید دستخط خودکار ہو۔

یہاں سب سے آسان طریقہ ہے: ایک ریپر کلاس جو کسی بھی کال ایبل ٹول فنکشن کو لے اور اس کا رسید جاری کرنے والا ورژن واپس کرے۔ یہ کسی بھی ایجنٹ فریم ورک کے ساتھ مطابقت رکھتا ہے، بشمول مائیکروسافٹ ایجنٹ فریم ورک (`agent_framework.foundry`)۔

اگر آپ کے پاس مائیکروسافٹ فاؤنڈری پراجیکٹ سیٹ اپ نہیں ہے، تو نیچے کا لوکل ماک پھر بھی اس طریقہ کار کو ظاہر کرتا ہے۔


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ انضمام

اوپر دیا گیا `ReceiptedTool` ریپر فریم ورک سے آزاد ہے۔ اسے مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ بنائے گئے ایجنٹ کے اندر استعمال کرنے کے لیے، ریپ کیا ہوا فنکشن ایک ٹول کے طور پر رجسٹر کریں۔ ایک خاکہ (آپ جعلی کو حقیقی مائیکروسافٹ فاؤنڈری ٹول رجسٹریشن سے بدلیں گے):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ایجنٹ فریم ورک کو رسیدوں کے بارے میں کچھ جاننے کی ضرورت نہیں ہے۔ رسید پر دستخط ٹول کے گرد لپٹا ہوا ہے، فریم ورک میں شامل نہیں کیا گیا۔ اس طرح آپ ایجنٹ کو دوبارہ لکھے بغیر موجودہ ایجنٹ کوڈ میں ماخذ شامل کرتے ہیں۔


## خلاصہ اور توسیعی چیلنج

آپ نے کیا:

- ایک Ed25519 کی جوڑی بنائی۔
- ایک ایجنٹ ٹول کال کے لیے رسید تیار کی اور دستخط کیں۔
- رسید کو صرف اجرا کرنے والے کی عوامی کلید استعمال کرتے ہوئے آف لائن تصدیق کی، جو رسید سے آزادانہ طور پر حاصل کی گئی تھی۔
- ایک رسید میں چھیڑ چھاڑ کی اور تصدیق کی ناکامی دیکھی۔
- تین رسیدوں کی ہیش چین والا سلسلہ تیار کیا۔
- زنجیر کے وسط میں چھیڑ چھاڑ کی اور دستخط ناکامی اور چین-لنک ناکامی دونوں دیکھی۔
- ایک ٹول فنکشن کو خودکار رسید دستخط کے ساتھ لپیٹا۔

**توسیعی چیلنج۔** رسید کے اسکیمہ میں `request_id` فیلڈ (منتشر ٹریسنگ کے لیے UUID) شامل کریں۔ `make_receipt` کو اپ ڈیٹ کریں تاکہ یہ شامل ہو، اور تصدیق کریں کہ رسیدیں اب بھی مکمل طور پر درست ہیں۔ پھر دستخط کے بعد اس فیلڈ میں ترمیم کریں اور تصدیق کی ناکامی کی تصدیق کریں۔ یہ آپ کو مجبور کرتا ہے کہ آپ داخلی طور پر سمجھیں کہ ہر بائٹ کی کینیونیکل انکوڈنگ دستخط میں کس طرح حصہ ڈالتا ہے۔

**اہم حد بندی۔** ایک مخصوص کلید اور چین سیاق و سباق کے تحت، رسیدیں دستخط کی نسبت، مواد کی سالمیت، اور پیش کردہ سابقہ لنک کو ثابت کرتی ہیں۔ یہ عمل درآمد، ماخذ حقیقت، معتبر وقت، مکمل پن، پالیسی جائزہ، یا درستگی ثابت نہیں کرتی ہیں۔ رسیدیں بنیاد ہیں۔ گورننس وہ نظام ہے جو آپ اس کے اوپر بناتے ہیں۔

اس حد بندی کو ذہن میں رکھتے ہوئے سبق کی README دوبارہ پڑھیں۔ ٹیموں کی سب سے عام غلطی یہ ہے کہ وہ یہ فرض کر لیتی ہیں کہ "ہمارے پاس رسیدیں ہیں" کا مطلب ہے "ہم مشتمل ہیں۔" ایسا نہیں ہے۔ رسیدیں ایجنٹ کے رویے کو قابل جانچ بناتی ہیں۔ یہ اسے درست نہیں بناتیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
